# データ探索と前処理

砲撃データの構造を確認し、パイプラインの Phase 1（境界抽出）を手動で実行します。

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fusou_formula.data_loader import DataLoader
from fusou_formula.phase1_boundary import BoundaryExtractor

## 1. データのロード

合成データを使用してパイプラインの動作を確認します。
実データがある場合は `load_hougeki_with_stats()` に差し替えてください。

In [ ]:
loader = DataLoader()

# 合成データ（既知の式 firepower * 1.5 + 5 + noise）
dataset = loader.create_synthetic(
    n_samples=3000,
    formula_str='floor(firepower * 1.5 + 5)',
    noise_range=(0.0, 0.999),
)

df = dataset.df
print(f'Samples: {len(df)}')
print(f'Target:  {dataset.target_col}')
print(f'Features: {dataset.feature_cols}')
df.head(10)

## 2. データの分布を確認

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(dataset.feature_cols[:3]):
    axes[i].scatter(df[col], df[dataset.target_col], alpha=0.3, s=5)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel(dataset.target_col)
    axes[i].set_title(f'{col} vs {dataset.target_col}')

plt.tight_layout()
plt.show()

## 3. Phase 1: 境界抽出

同一条件のデータをグルーピングし、最大値（または最小値）を取ります。
これによりランダム成分の上限/下限が推定できます。

In [ ]:
extractor = BoundaryExtractor(boundary='max', min_group_size=2)
result = extractor.fit(df, target_col=dataset.target_col)

print(f'Original rows:  {result.n_original}')
print(f'Groups:         {result.n_groups}')
print(f'Boundary mode:  {result.boundary}')
print()
result.df.head(10)

## 4. ランダム幅の推定

In [ ]:
# max - min で各グループのランダムレンジを推定
ranges = []
for key, info in result.range_info.items():
    ranges.append(info.max_value - info.min_value)

ranges = np.array(ranges)
print(f'Mean range:   {np.mean(ranges):.3f}')
print(f'Median range: {np.median(ranges):.3f}')
print(f'Max range:    {np.max(ranges):.3f}')

plt.figure(figsize=(8, 3))
plt.hist(ranges, bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Range (max - min)')
plt.ylabel('Count')
plt.title('Distribution of random range per group')
plt.tight_layout()
plt.show()

## まとめ

- 同一入力で複数の出力がある場合、`max` 境界を使うことでランダム成分の上限値を抽出
- ランダム幅の分布から `random_range_min` / `random_range_max` を設定
- 次のステップ: `02_hypothesis_testing.ipynb` で Phase 2〜5 を実行